In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import InterContiNetTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

### Task Incremental Learning

In [107]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_fully_connected_model(
    device="cuda",
    output_dim=10,
    input_dim=28 * 28,
    hidden_dim=400,
    head=head,
    dense_layers=2,
    seed=SEED
)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

interval_trainer = InterContiNetTrainer(
    model,
    min_acc_limit=0.9,
    paradigm="TIL",
    seed=SEED
)

lr = 0.01
batch_size = 128
epochs = 30
lid_lr = 100
for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        context_id=i,
        lr=lr,
        batch_size=batch_size,
        epochs=epochs,
    )
    interval_trainer.test(test_tasks, context_list=list(range(5)))
    if i == len(train_tasks) - 1:
        continue
    interval_trainer.compute_rashomon_set(
        test, context_id=i, lr=lid_lr, batch_size=256, epochs=1000
    )


Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


Training Epochs: 100%|████████████████████████████████| 30/30 [00:28<00:00,  1.06it/s, val_loss=0.0306, val_acc=0.9935]


Test Results: [(0.0627, 0.987), (41.9291, 0.5073), (14.8644, 0.4762), (33.7181, 0.4868), (24.3871, 0.476)]
LID size: 318010.0000.


  0%|                                               | 0/1000 [00:00<?, ?it/s, loss=23.6289, acc=0.9492, size=104901.18]


LID size: 104901.1777 with certificate of 0.94921875.


...

Test Results: [(0.0629, 0.9865), (0.3871, 0.8582), (0.2897, 0.9326), (0.3086, 0.872), (24.4006, 0.476)]
LID size: 70175.8918.


  0%|                                         | 1/1000 [00:00<03:54,  4.27it/s, loss=0.2548, acc=0.9062, size=55685.92]


LID size: 55685.9236 with certificate of 0.90625.


Training Epochs: 100%|████████████████████████████████| 30/30 [00:26<00:00,  1.14it/s, val_loss=0.4207, val_acc=0.8040]


Test Results: [(0.0629, 0.9865), (0.3871, 0.8582), (0.2903, 0.9326), (0.3086, 0.872), (0.4019, 0.817)]


### Domain Incremental Learning

In [108]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [109]:
SEED = 2

train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_fully_connected_model(
    device="cuda",
    output_dim=2,
    input_dim=28 * 28,
    hidden_dim=400,
    seed=SEED,
    dense_layers=2,
)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

interval_trainer = InterContiNetTrainer(
    model,
    min_acc_increment=0.1,
    min_acc_limit=0.8,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
    seed=SEED,
)

lr = 0.01
batch_size = 128
epochs = 30
lid_lr = 100
for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        lr=lr,
        batch_size=batch_size,
        epochs=epochs,
    )
    results = interval_trainer.test(test_tasks)
    print(sum([res[1] for res in results]) / 5)
    if i == len(train_tasks) - 1:
        continue
    target_acc = min(
        max(results[i][1] - interval_trainer.min_acc_increment, results[i][1] / 2),
        interval_trainer.min_acc_limit,
    )
    interval_trainer.min_acc_limit = target_acc
    interval_trainer.compute_rashomon_set(test, lr=lid_lr)

Tasks: [[6, 7], [8, 9], [2, 5], [0, 3], [1, 4]]


Training Epochs: 100%|████████████████████████████████| 30/30 [00:28<00:00,  1.06it/s, val_loss=0.0009, val_acc=0.9996]


Test Results: [(0.0322, 0.996), (4.4045, 0.6586), (4.3184, 0.6897), (4.2471, 0.6844), (3.3398, 0.5626)]
0.7182599999999999
LID size: 314802.0000.


  0%|                                                 | 0/500 [00:00<?, ?it/s, loss=82.8715, acc=0.8984, size=98138.75]


LID size: 98138.7500 with certificate of 0.8984375.


...

Test Results: [(0.0317, 0.9955), (4.5672, 0.6515), (4.1495, 0.7006), (4.38, 0.6789), (3.3242, 0.564)]
0.7181
LID size: 74484.2188.


  0%|                                                 | 0/500 [00:00<?, ?it/s, loss=18.9090, acc=0.6719, size=73985.70]


LID size: 73985.7031 with certificate of 0.671875.


Training Epochs: 100%|████████████████████████████████| 30/30 [00:29<00:00,  1.03it/s, val_loss=3.5481, val_acc=0.5706]


Test Results: [(0.032, 0.9955), (4.6299, 0.648), (4.1258, 0.7017), (4.3819, 0.6784), (3.2815, 0.5664)]
0.718


### Class Incremental Training

In [110]:
SEED = 0

train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_fully_connected_model(
    device="cuda", output_dim=10, input_dim=28 * 28, hidden_dim=400, seed=SEED
)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

interval_trainer = InterContiNetTrainer(
    model, min_acc_limit=0.8, paradigm="CIL", seed=SEED
)

lr = 0.01
batch_size = 128
epochs = 30
lid_lr = 100
for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    interval_trainer.train(
        train,
        val,
        lr=lr,
        batch_size=batch_size,
        epochs=epochs,
    )
    results = interval_trainer.test(test_tasks)
    print(sum([res[1] for res in results]) / 5)
    if i == len(train_tasks) - 1:
        continue
    target_acc = min(
        max(results[i][1] - interval_trainer.min_acc_increment, results[i][1] / 2),
        interval_trainer.min_acc_limit,
    )
    interval_trainer.min_acc_limit = target_acc
    interval_trainer.compute_rashomon_set(test, lr=lid_lr)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


Training Epochs: 100%|████████████████████████████████| 30/30 [00:26<00:00,  1.12it/s, val_loss=0.0229, val_acc=0.9939]


Test Results: [(0.0624, 0.9765), (16.7636, 0.0), (12.7251, 0.0), (15.012, 0.0), (14.4346, 0.0)]
0.1953
LID size: 7850.0000.


  0%|                                                   | 0/500 [00:00<?, ?it/s, loss=0.3384, acc=0.9375, size=3208.61]


LID size: 3208.6145 with certificate of 0.9375.


...

Test Results: [(0.0669, 0.975), (14.8029, 0.0146), (10.9254, 0.0517), (12.0919, 0.0091), (13.9235, 0.0)]
0.21008
LID size: 2532.0220.


  0%|                                                 | 0/500 [00:00<?, ?it/s, loss=12.1079, acc=0.02344, size=2100.14]


LID size: 2100.1406 with certificate of 0.0234375.


Training Epochs: 100%|███████████████████████████████| 30/30 [00:24<00:00,  1.21it/s, val_loss=13.1960, val_acc=0.0027]


Test Results: [(0.0692, 0.9745), (14.8151, 0.0141), (10.9245, 0.0521), (12.0913, 0.0091), (13.6958, 0.0011)]
0.21018000000000003


### Class Incremental Learning + Regulariser

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_fully_connected_mnist_model(device="cuda", hidden_dim=400)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [ ]:
unbias = UnbiasRegulariser(lmbd=0)
l2 = L2Regulariser(lmbd=1)
regulariser = MultiRegulariser([l2, unbias])

trainer = SimpleTrainer(model)
trainer.train(
    train_tasks[0], val_tasks[0], epochs=1, batch_size=256, regulariser=regulariser
)
trainer.test(test_tasks[0:1], regulariser=regulariser)

Training Epochs: 100%|███████████████████████████████████████████| 1/1 [00:05<00:00,  5.81s/it, val_loss=3.02, val_acc=0.979]


Test Results: [(2.9894, 0.9874)]


[(2.9894, 0.9874)]

In [ ]:
interval_trainer = InterContiNetTrainer(
    trainer.model,
    min_acc_limit=0.9,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256)
    test_vals = interval_trainer.test(test_tasks[0 : i + 1])
    if not test_vals[-1][1]:
        break
    if i < len(train_tasks) - 2:
        target_acc = min(
            max(test_vals[-1][1] - interval_trainer.min_acc_increment, 0),
            interval_trainer.min_acc_limit,
        )
        interval_trainer.min_acc_limit = target_acc
        interval_trainer.compute_rashomon_set(test)

 54%|█████████████████████████████▏                        | 54/100 [00:17<00:14,  3.18it/s, max loss=0.2071, min acc=0.9375]


LID size: 5216.6416


Training Epochs: 100%|███████████████████████████████████████| 5/5 [00:27<00:00,  5.57s/it, val_loss=12.0082, val_acc=0.0000]


Test Results: [(0.0241, 0.994), (12.3367, 0.0)]
